# 05 — Vehicle Routing Optimisation
Solve a Capacitated VRP using Google OR-Tools and compare against baseline routes.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.optimization import build_distance_matrix, solve_vrp, compare_routes

%matplotlib inline


## 5.1 Load location data

In [ ]:
# Replace with actual lat/lon from your dataset.
# Index 0 must be the depot; indices 1…N are customer/delivery locations.

demo_locations = [
    (17.3850, 78.4867),  # 0: Depot (Hyderabad)
    (17.4400, 78.4900),  # 1
    (17.3600, 78.5500),  # 2
    (17.4200, 78.4200),  # 3
    (17.3100, 78.5200),  # 4
    (17.4800, 78.3900),  # 5
    (17.3700, 78.4100),  # 6
]

demands          = [0, 10, 15, 20, 12, 8, 18]   # depot demand = 0
vehicle_caps     = [60, 60]                       # 2 vehicles, capacity 60 each
vehicle_names    = ['Vehicle A', 'Vehicle B']

distance_matrix = build_distance_matrix(demo_locations)
print("Distance matrix (metres):")
for row in distance_matrix:
    print(row)


## 5.2 Solve the VRP

In [ ]:
result = solve_vrp(
    distance_matrix=distance_matrix,
    demands=demands,
    vehicle_capacities=vehicle_caps,
    depot=0,
    time_limit_seconds=30,
    verbose=True,
)

print("\nStatus:", result['status'])
print("Total distance:", result['total_distance'])
if result['dropped_nodes']:
    print("Dropped nodes:", result['dropped_nodes'])


## 5.3 Visualise optimised routes

In [ ]:
import matplotlib.patches as mpatches

lats = [loc[0] for loc in demo_locations]
lons = [loc[1] for loc in demo_locations]
colors = plt.cm.tab10.colors

fig, ax = plt.subplots(figsize=(10,8))

# Plot all nodes
ax.scatter(lons, lats, s=120, zorder=5, c='black')
for i, (lat, lon) in enumerate(demo_locations):
    label = 'Depot' if i == 0 else f'Stop {i}'
    ax.annotate(label, (lon, lat), textcoords='offset points',
                xytext=(5,5), fontsize=9)

# Draw routes
for v_idx, route in enumerate(result['routes']):
    color = colors[v_idx % len(colors)]
    for step in range(len(route)-1):
        n_from, n_to = route[step], route[step+1]
        ax.annotate(
            '', xy=(lons[n_to], lats[n_to]),
            xytext=(lons[n_from], lats[n_from]),
            arrowprops=dict(arrowstyle='->', color=color, lw=2)
        )

patches = [mpatches.Patch(color=colors[i], label=vehicle_names[i] if i < len(vehicle_names) else f'Vehicle {i}')
           for i in range(len(result['routes']))]
ax.legend(handles=patches, loc='lower right')
ax.set_title('Optimised Delivery Routes')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.savefig('../outputs/figures/optimised_routes.png', dpi=150)
plt.show()


## 5.4 Before / After comparison

In [ ]:
# Hypothetical baseline for illustration
baseline_dist   = result['total_distance'] * 1.25  # assume 25% longer
baseline_late   = 18.5
optimized_late  = 9.2

comparison = compare_routes(
    baseline_distance=baseline_dist,
    optimized_distance=result['total_distance'],
    baseline_late_rate=baseline_late,
    optimized_late_rate=optimized_late,
)
print(comparison.to_string(index=False))


## 5.5 Extending to real data

Replace `demo_locations` with coordinates from your customer/location table. Build the distance matrix from a real travel-time API (e.g. Google Maps, OpenRouteService) for production-quality routing. The OR-Tools solver scales to hundreds of stops with a reasonable time limit.